# Impute solvents from a processed NPZ

This notebook loads a processed structure `.npz`, optionally strips existing solvents, imputes additional solvents with `Structure.impute_solvents(...)`, and writes the result as an mmCIF file.

Edit the config cell below, then run the notebook top to bottom.

In [7]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import numpy as np


BOLTZGEN_SRC = Path("/data/rbg/users/gloriama/dev/foldeverything/src")
NOTEBOOK_ROOT = Path("/data/rbg/users/gloriama/dev/water_conservation")

if str(BOLTZGEN_SRC) not in sys.path:
    sys.path.insert(0, str(BOLTZGEN_SRC))
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

from boltzgen.data.data import Structure
from boltzgen.data.write.mmcif import to_mmcif
from gloria_hbond_helpers import (
    gloria_get_solvent_hbond_counts,
    gloria_get_solvent_hbond_counts_and_mask,
    gloria_get_solvent_hbond_mask,
)
from gloria_solvent_filters import (
    gloria_remove_low_b_factor_solvents,
    gloria_remove_weak_solvents,
)
from impute_solvents_with_num_hbonds import impute_solvents_with_num_hbonds


def resolve_npz_path(pdb_id: str, npz_root: Path, npz_path: str | None = None) -> Path:
    if npz_path:
        return Path(npz_path)
    return Path(npz_root) / f"{pdb_id.lower()}.npz"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Config
PDB_ID = "7zzh"
NPZ_ROOT = Path("/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures")
NPZ_PATH = None  # Set to an explicit file path to override PDB_ID + NPZ_ROOT.
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs"

MIN_SOLVENTS = 20
STRIP_SOLVENTS_BEFOREHAND = True
ALLOW_OVERLAP_WITH_REAL_SOLVENTS = False
ONE_SOLVENT_PER_CHAIN = False
INTERACTION_MIN_DIST = 2.5
MAX_SAMPLE_ATTEMPTS = 10000


In [4]:
npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if not npz_path.exists():
    raise FileNotFoundError(f"Could not find NPZ input: {npz_path}")

structure = Structure.load(npz_path)
loaded_solvents = structure.count_solvents()

if STRIP_SOLVENTS_BEFOREHAND:
    structure = structure.remove_solvents()

post_strip_solvents = structure.count_solvents()
structure = impute_solvents_with_num_hbonds(
    structure,
    one_solvent_per_chain=ONE_SOLVENT_PER_CHAIN,
    min_solvents=MIN_SOLVENTS,
    interaction_min_dist=INTERACTION_MIN_DIST,
    max_sample_attempts=MAX_SAMPLE_ATTEMPTS,
    allow_overlap_with_real_solvents=ALLOW_OVERLAP_WITH_REAL_SOLVENTS,
)
final_solvents = structure.count_solvents()

output_path = output_dir / f"{PDB_ID.lower()}_imputed_first_stripped.cif"
output_path.write_text(to_mmcif(structure))

print(f"Input NPZ: {npz_path}")
print(f"Loaded solvent count: {loaded_solvents}")
print(f"Post-strip solvent count: {post_strip_solvents}")
print(f"Final solvent count: {final_solvents}")
print(f"Wrote CIF to: {output_path}")


sample_attempts=10003 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=0th water.
sample_attempts=10001 until a n_function(sample_attempts)=1 h-bond thing was found, for the impute_idx=1th water.


KeyboardInterrupt: 

In [ ]:
# Example helper calls
example_structure = Structure.load(npz_path)
example_structure = example_structure.to_one_solvent_per_chain(example_structure)

gloria_hbond_counts = gloria_get_solvent_hbond_counts(example_structure)
(
    gloria_hbond_counts_full,
    gloria_present_solvent_chain_mask,
) = gloria_get_solvent_hbond_counts_and_mask(example_structure)
gloria_keep_solvent_mask = gloria_get_solvent_hbond_mask(
    example_structure,
    min_hbonds=2,
)

gloria_weak_filtered_structure = gloria_remove_weak_solvents(
    example_structure,
    min_hbonds=2,
)
gloria_bfactor_filtered_structure = gloria_remove_low_b_factor_solvents(
    example_structure,
    n_keep=10,
)

print(f"Present solvent chains: {gloria_present_solvent_chain_mask.sum()}")
print(f"Per-chain hbond counts shape: {gloria_hbond_counts.shape}")
print(
    "Counts agree across helper variants:",
    (gloria_hbond_counts == gloria_hbond_counts_full).all(),
)
print(f"Chains meeting min_hbonds=2: {gloria_keep_solvent_mask.sum()}")
print(
    f"Solvents after gloria_remove_weak_solvents: {gloria_weak_filtered_structure.count_solvents()}"
)
print(
    f"Solvents after gloria_remove_low_b_factor_solvents(n_keep=10): {gloria_bfactor_filtered_structure.count_solvents()}"
)

In [21]:
from filter_solvent_clashes import filter_solvent_clashes
from impute_solvents_from_triples import (
    find_atom_triples_for_three_hbonds,
    impute_solvents_from_atom_triples,
)

PDB_ID = "1ubq"
TRIPLE_HBOND_PAIR_MAX_DIST = 8.0

triple_npz_path = resolve_npz_path(PDB_ID, NPZ_ROOT, NPZ_PATH)
triple_output_dir = Path(OUTPUT_DIR)
triple_output_dir.mkdir(parents=True, exist_ok=True)
triple_max_dist_tag = str(TRIPLE_HBOND_PAIR_MAX_DIST).replace(".", "p")

triple_input_structure = Structure.load(triple_npz_path).remove_solvents()
triple_atom_triples = find_atom_triples_for_three_hbonds(
    triple_input_structure,
    max_pair_dist=TRIPLE_HBOND_PAIR_MAX_DIST,
)
triple_imputed_structure = impute_solvents_from_atom_triples(
    triple_input_structure,
    one_solvent_per_chain=True,
    max_pair_dist=TRIPLE_HBOND_PAIR_MAX_DIST,
)
triple_filtered_structure = filter_solvent_clashes(
    triple_imputed_structure,
    min_dist=INTERACTION_MIN_DIST,
)

triple_output_path = triple_output_dir / (
    f"{PDB_ID.lower()}_all_triples_stripped_filtered_maxdist_{triple_max_dist_tag}.cif"
)
triple_output_path.write_text(to_mmcif(triple_filtered_structure))

print(f"Input NPZ: {triple_npz_path}")
print(f"Triple max pair distance: {TRIPLE_HBOND_PAIR_MAX_DIST}")
print(f"Triple count: {len(triple_atom_triples)}")
print(f"Placed waters before filtering: {triple_imputed_structure.count_solvents()}")
print(f"Placed waters after filtering: {triple_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {triple_output_path}")

Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Triple max pair distance: 8.0
Triple count: 550
Placed waters before filtering: 550
Placed waters after filtering: 9
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_all_triples_stripped_filtered_maxdist_8p0.cif


In [16]:
default_pdb_id = "1ubq"
default_npz_path = resolve_npz_path(default_pdb_id, NPZ_ROOT)
default_output_dir = Path(OUTPUT_DIR)
default_output_dir.mkdir(parents=True, exist_ok=True)

default_structure = Structure.load(default_npz_path)
default_output_path = default_output_dir / f"{default_pdb_id}_default.cif"
default_output_path.write_text(to_mmcif(default_structure))

print(f"Input NPZ: {default_npz_path}")
print(f"Wrote CIF to: {default_output_path}")

Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_default.cif


In [17]:
weak_pdb_id = "1ubq"
weak_npz_path = resolve_npz_path(weak_pdb_id, NPZ_ROOT)
weak_output_dir = Path(OUTPUT_DIR)
weak_output_dir.mkdir(parents=True, exist_ok=True)
weak_structure = Structure.load(weak_npz_path)
weak_structure = weak_structure.to_one_solvent_per_chain(weak_structure)
weak_filtered_structure = gloria_remove_weak_solvents(weak_structure)
weak_output_path = weak_output_dir / f"{weak_pdb_id}_weak_filtered.cif"
weak_output_path.write_text(to_mmcif(weak_filtered_structure))
print(f"Input NPZ: {weak_npz_path}")
print(f"Original solvent count: {weak_structure.count_solvents()}")
print(f"Filtered solvent count: {weak_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak_output_path}")


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Original solvent count: 58
Filtered solvent count: 17
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_weak_filtered.cif


In [18]:
weak3_pdb_id = "1ubq"
weak3_npz_path = resolve_npz_path(weak3_pdb_id, NPZ_ROOT)
weak3_output_dir = Path(OUTPUT_DIR)
weak3_output_dir.mkdir(parents=True, exist_ok=True)
weak3_structure = Structure.load(weak3_npz_path)
weak3_structure = weak3_structure.to_one_solvent_per_chain(weak3_structure)
weak3_filtered_structure = gloria_remove_weak_solvents(
    weak3_structure,
    min_hbonds=3,
)
weak3_output_path = weak3_output_dir / f"{weak3_pdb_id}_weak_filtered_min3.cif"
weak3_output_path.write_text(to_mmcif(weak3_filtered_structure))
print(f"Input NPZ: {weak3_npz_path}")
print(f"Original solvent count: {weak3_structure.count_solvents()}")
print(f"Filtered solvent count (min_hbonds=3): {weak3_filtered_structure.count_solvents()}")
print(f"Wrote CIF to: {weak3_output_path}")


Input NPZ: /data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures/1ubq.npz
Original solvent count: 58
Filtered solvent count (min_hbonds=3): 3
Wrote CIF to: /data/rbg/users/gloriama/dev/water_conservation/outputs/1ubq_weak_filtered_min3.cif
